In [1]:
#!/usr/bin/env python3
"""
DGH-XH: Delhi AQI (CPCB + ERA5)
==================================================
Dataset : CPCB Delhi + ERA5-Land (user's Kaggle dataset)
          /kaggle/input/datasets/kumaran13885/aqi-era5/windowed/delhi/
Target  : PM2.5 (μg/m³)
Stations: 30 (dense network)
"""

# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import os, warnings, joblib, json
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)
import xgboost as xgb
from xgboost import XGBRegressor
from IPython.display import display
warnings.filterwarnings("ignore")

# ============================================================
# CELL 2 — CONFIG
# ============================================================
CITY      = "delhi"
BASE_PATH = "/kaggle/input/datasets/kumaran13885/aqi-era5/windowed"
CITY_PATH = os.path.join(BASE_PATH, CITY)

OUT_DIR     = "dghxh_delhi_aqi"
L           = 24
H_LIST      = [1, 3, 6, 12, 24]
TAU_LIST    = [0, 1, 2, 3, 4, 6]
VALID_FRAC  = 0.20
GRAPH_K     = 4
RUN_TUNED   = True
JSO_POP     = 6
JSO_ITERS   = 8
RANDOM_SEED = 1

ASTRA_FEATURES = [
    "pm25","pm10","no2","so2",
    "temp_2m","dewpoint_2m","surface_pressure","u10","v10",
    "hour_sin","hour_cos","dow_sin","dow_cos",
    "doy_sin","doy_cos","month_sin","month_cos"
]
FEATURES      = ["pm25","temp_2m","dewpoint_2m","surface_pressure","u10","v10"]
FEAT_IDX      = [ASTRA_FEATURES.index(f) for f in FEATURES]
TARGET_COL    = "pm25"
PM25_POLL_IDX = 0
H_TO_IDX      = {1:0, 3:1, 6:2, 12:3, 24:4}



# CELL 4 — ALL SHARED HELPER FUNCTIONS (identical to Chennai)
# ============================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2.0*R*np.arcsin(np.sqrt(a))

def bearing_radians(lat1, lon1, lat2, lon2):
    lat1,lon1,lat2,lon2 = map(np.radians,[lat1,lon1,lat2,lon2])
    dlon = lon2-lon1
    y = np.sin(dlon)*np.cos(lat2)
    x = np.cos(lat1)*np.sin(lat2)-np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    return np.arctan2(y, x)

def flatten_window_per_node(X):
    S,Lx,N,F = X.shape
    return X.transpose(0,2,1,3).reshape(S*N, Lx*F)

def compute_tail_metrics(y_true, y_pred, percentiles=(90,95,99)):
    rows = []
    for p in percentiles:
        thr = np.percentile(y_true, p)
        idx = y_true >= thr
        if idx.sum() == 0:
            rows.append((p, thr, np.nan, np.nan, 0))
        else:
            rows.append((p, thr,
                mean_absolute_error(y_true[idx], y_pred[idx]),
                np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])),
                int(idx.sum())))
    return rows

def compute_mfb_nmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mfb  = np.mean(2.0*(y_pred-y_true)/(y_pred+y_true+1e-8))
    nmse = np.sum((y_pred-y_true)**2)/(np.sum(y_pred*y_true)+1e-8)
    return mfb, nmse

def build_fill_values_from_train_timeline(X_train_timeline):
    fill_values = np.nanmedian(X_train_timeline, axis=0)
    global_fill = np.nanmedian(
        X_train_timeline.reshape(-1, X_train_timeline.shape[-1]), axis=0)
    global_fill = np.where(np.isnan(global_fill), 0.0, global_fill)
    for n in range(fill_values.shape[0]):
        for f in range(fill_values.shape[1]):
            if np.isnan(fill_values[n,f]):
                fill_values[n,f] = global_fill[f]
    return np.where(np.isnan(fill_values), 0.0, fill_values).astype(np.float32)

def impute_windows(X, fill_values):
    X_imp = X.copy().astype(np.float32)
    S,Lx,N,F = X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                series = pd.Series(X_imp[s,:,n,f], dtype="float32")
                series = series.ffill().bfill()
                arr = series.to_numpy(dtype=np.float32)
                if np.isnan(arr).any():
                    arr = np.where(np.isnan(arr), fill_values[n,f], arr)
                X_imp[s,:,n,f] = arr
    return X_imp

def build_nodes_from_coords(station_ids, coord_dict):
    """Build nodes DataFrame from station_id → (lat, lon) dict."""
    rows = [{"station_id": sid,
             "lat": coord_dict[sid][0],
             "lon": coord_dict[sid][1],
             "node_id": i}
            for i, sid in enumerate(station_ids)]
    return pd.DataFrame(rows)

def build_edges_from_nodes(nodes_df, k=4):
    coords = nodes_df[["lat","lon"]].to_numpy(dtype=float)
    N = len(coords)
    D = np.zeros((N,N))
    for i in range(N):
        D[i,:] = haversine_km_vec(
            coords[i,0], coords[i,1], coords[:,0], coords[:,1])
    sigma = np.median(D[D>0])
    edges = []
    for i in range(N):
        nn = np.argsort(D[i])[1:min(k+1,N)]
        for j in nn:
            w = np.exp(-(D[i,j]**2)/(2.0*sigma**2))
            edges.append((i, j, float(w), float(D[i,j])))
    return pd.DataFrame(edges, columns=["src","dst","w_dist","dist_km"])

def build_static_adj(nodes_df, k=4):
    coords = nodes_df[["lat","lon"]].to_numpy(dtype=float)
    N = len(coords)
    D = np.zeros((N,N))
    for i in range(N):
        D[i,:] = haversine_km_vec(
            coords[i,0], coords[i,1], coords[:,0], coords[:,1])
    sigma = np.median(D[D>0])
    A = np.zeros((N,N), dtype=np.float32)
    for i in range(N):
        nn = np.argsort(D[i])[1:min(k+1,N)]
        for j in nn:
            A[i,j] = np.exp(-(D[i,j]**2)/(2.0*sigma**2))
    rs = A.sum(axis=1, keepdims=True)
    return np.divide(A, rs, out=np.zeros_like(A), where=rs>0)

def build_dynamic_adj(u_t, v_t, ctx, alpha=4.0, eps=0.05):
    theta_w = np.arctan2(v_t[ctx["src"]], u_t[ctx["src"]])
    align   = np.cos(theta_w - ctx["edge_bearing"])
    gate    = eps + (1.0-eps)/(1.0+np.exp(-alpha*align))
    w_dyn   = ctx["w_dist"] * gate
    A = np.zeros((ctx["N"], ctx["N"]), dtype=np.float32)
    A[ctx["src"], ctx["dst"]] = w_dyn.astype(np.float32)
    rs = A.sum(axis=1, keepdims=True)
    return np.divide(A, rs, out=np.zeros_like(A), where=rs>0)

def make_graph_features_dynamic(X, ctx, tau=0, alpha=4.0, eps=0.05):
    S,Lx,N,F = X.shape
    Z = np.zeros((S,N,2*F), dtype=np.float32)
    for s in range(S):
        x_now = X[s,-1]
        x_lag = X[s,-1-tau] if tau > 0 else x_now
        A = build_dynamic_adj(x_now[:,ctx["u_idx"]], x_now[:,ctx["v_idx"]],
                              ctx, alpha=alpha, eps=eps)
        Z[s] = np.concatenate([x_now, A @ x_lag], axis=-1)
    return Z

def make_graph_features_static(X, A_static, tau=0):
    S,Lx,N,F = X.shape
    Z = np.zeros((S,N,2*F), dtype=np.float32)
    for s in range(S):
        x_now = X[s,-1]
        x_lag = X[s,-1-tau] if tau > 0 else x_now
        Z[s] = np.concatenate([x_now, A_static @ x_lag], axis=-1)
    return Z

def summarize_regression(y_true, y_pred):
    return {"MAE":  float(mean_absolute_error(y_true, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2":   float(r2_score(y_true, y_pred))}

def train_val_split_time_order(X, Y, M, times, frac=0.2):
    n = X.shape[0]
    val_size = max(1, int(np.ceil(frac*n)))
    idx_tr = np.arange(0, n-val_size)
    idx_va = np.arange(n-val_size, n)
    tr = {"X":X[idx_tr],"Y":Y[idx_tr],"M":M[idx_tr],"times":times[idx_tr]}
    va = {"X":X[idx_va],"Y":Y[idx_va],"M":M[idx_va],"times":times[idx_va]}
    return tr, va

def fit_flat_regressor(model, Xtr, Ytr, Mtr, Xte, Yte, Mte):
    Xtr2 = flatten_window_per_node(Xtr)
    Xte2 = flatten_window_per_node(Xte)
    ytr = Ytr.reshape(-1); yte = Yte.reshape(-1)
    mtr = (Mtr.reshape(-1)>0.5); mte = (Mte.reshape(-1)>0.5)
    model.fit(Xtr2[mtr], ytr[mtr])
    return {"y_true":yte[mte], "y_pred":model.predict(Xte2)[mte], "model":model}

def fit_static_graph_regressor(model, Xtr, Ytr, Mtr, Xte, Yte, Mte, A_static, tau):
    Ztr = make_graph_features_static(Xtr, A_static, tau=tau)
    Zte = make_graph_features_static(Xte, A_static, tau=tau)
    Xtr2 = Ztr.reshape(-1, Ztr.shape[-1]); Xte2 = Zte.reshape(-1, Zte.shape[-1])
    ytr = Ytr.reshape(-1); yte = Yte.reshape(-1)
    mtr = (Mtr.reshape(-1)>0.5); mte = (Mte.reshape(-1)>0.5)
    model.fit(Xtr2[mtr], ytr[mtr])
    return {"y_true":yte[mte], "y_pred":model.predict(Xte2)[mte], "model":model}

def fit_huber_graph(Xtr, Ytr, Mtr, Xte, Yte, Mte, ctx,
                    tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4):
    Ztr = make_graph_features_dynamic(Xtr, ctx, tau=tau, alpha=alpha, eps=eps)
    Zte = make_graph_features_dynamic(Xte, ctx, tau=tau, alpha=alpha, eps=eps)
    Xtr2 = Ztr.reshape(-1, Ztr.shape[-1]); Xte2 = Zte.reshape(-1, Zte.shape[-1])
    ytr = Ytr.reshape(-1); yte = Yte.reshape(-1)
    mtr = (Mtr.reshape(-1)>0.5); mte = (Mte.reshape(-1)>0.5)
    huber = Pipeline([("sc", StandardScaler()),
                      ("h", HuberRegressor(epsilon=1.35, alpha=huber_alpha, max_iter=500))])
    huber.fit(Xtr2[mtr], ytr[mtr])
    ptr_all = huber.predict(Xtr2)
    pte_all = huber.predict(Xte2)
    return {"y_true":yte[mte], "y_pred":pte_all[mte],
            "pred_train_all":ptr_all, "pred_test_all":pte_all,
            "y_train_all":ytr, "y_test_all":yte,
            "mask_train":mtr, "mask_test":mte, "model":huber}

def fit_huber_hybrid(Xtr, Ytr, Mtr, Xte, Yte, Mte, ctx,
                     tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4,
                     xgb_params=None, num_boost_round=400, seed=42,
                     w_mid=2.0, w_danger=5.0, w_tail=10.0):
    base = fit_huber_graph(Xtr, Ytr, Mtr, Xte, Yte, Mte, ctx,
                           tau=tau, alpha=alpha, eps=eps, huber_alpha=huber_alpha)
    ytr_all = base["y_train_all"]; mtr = base["mask_train"]; mte = base["mask_test"]
    res_train = ytr_all - base["pred_train_all"]
    Xg_tr = np.asarray(flatten_window_per_node(Xtr), dtype=np.float32)
    Xg_te = np.asarray(flatten_window_per_node(Xte), dtype=np.float32)
    t90 = np.percentile(ytr_all[mtr], 90)
    t95 = np.percentile(ytr_all[mtr], 95)
    t99 = np.percentile(ytr_all[mtr], 99)
    w = np.ones_like(ytr_all[mtr], dtype=np.float32)
    w[ytr_all[mtr] >= t90] = w_mid
    w[ytr_all[mtr] >= t95] = w_danger
    w[ytr_all[mtr] >= t99] = w_tail
    dtrain = xgb.DMatrix(Xg_tr[mtr], label=res_train[mtr], weight=w)
    if xgb_params is None:
        xgb_params = {"objective":"reg:pseudohubererror","max_depth":6,"eta":0.05,
                      "subsample":0.80,"colsample_bytree":0.80,"lambda":1.0,
                      "tree_method":"hist","seed":seed,"verbosity":0}
    booster = xgb.train(xgb_params, dtrain, num_boost_round=int(num_boost_round))
    res_te_all = booster.predict(xgb.DMatrix(Xg_te))
    final = base["pred_test_all"].copy()
    final[mte] = final[mte] + res_te_all[mte]
    return {"y_true":base["y_test_all"][mte],
            "y_pred_graph":base["pred_test_all"][mte],
            "y_pred_final":final[mte],
            "gmodel":base["model"], "hmodel":booster}

def choose_best_tau_dynamic(tr, va, ctx, tau_list):
    best_tau, best_mae = None, np.inf
    rows = []
    for tau in tau_list:
        if tau >= tr["X"].shape[1]: continue
        res = fit_huber_graph(tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau)
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau":tau,"val_MAE":mae})
        if mae < best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(tr, va, A_static, tau_list):
    best_tau, best_mae = None, np.inf
    rows = []
    for tau in tau_list:
        if tau >= tr["X"].shape[1]: continue
        m = Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=500))])
        res = fit_static_graph_regressor(m,tr["X"],tr["Y"],tr["M"],
                                          va["X"],va["Y"],va["M"],A_static,tau)
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau":tau,"val_MAE":mae})
        if mae < best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params, tau, tr, va, ctx):
    (alpha,eps,log_ha,max_depth,eta,subsample,colsample,
     reg_lambda,nround,w_mid,w_danger,w_tail) = params
    hybrid = fit_huber_hybrid(
        tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau,
        alpha=float(alpha), eps=float(eps), huber_alpha=10**float(log_ha),
        xgb_params={"objective":"reg:pseudohubererror",
                    "max_depth":int(round(max_depth)),"eta":float(eta),
                    "subsample":float(subsample),"colsample_bytree":float(colsample),
                    "lambda":float(reg_lambda),"tree_method":"hist",
                    "seed":RANDOM_SEED,"verbosity":0},
        num_boost_round=int(round(nround)), seed=RANDOM_SEED,
        w_mid=float(w_mid), w_danger=float(w_danger), w_tail=float(w_tail))
    mae = mean_absolute_error(hybrid["y_true"], hybrid["y_pred_final"])
    thr = np.percentile(hybrid["y_true"], 99)
    idx = hybrid["y_true"] >= thr
    if idx.sum() > 0:
        mfb,_ = compute_mfb_nmse(hybrid["y_true"][idx], hybrid["y_pred_final"][idx])
        return mae + 10.0*abs(mfb)
    return mae

def jso_optimize(tr, va, ctx, tau, bounds, n_pop=12, iters=15, seed=1):
    np.random.seed(seed)
    dim = bounds.shape[0]
    pop = bounds[:,0] + np.random.rand(n_pop,dim)*(bounds[:,1]-bounds[:,0])
    fit = np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in pop])
    best_p = pop[np.argmin(fit)].copy()
    best_f = float(fit.min())
    print(f"  initial best={best_f:.4f}")
    for it in range(iters):
        c = 1.0 - it/max(iters,1)
        new_pop = pop.copy()
        for i in range(n_pop):
            if np.random.rand() < 0.5:
                cand = pop[i] + np.random.randn(dim)*c*0.1 + c*(best_p-pop[i])*np.random.rand(dim)
            else:
                j = np.random.randint(0,n_pop)
                cand = pop[i] + (pop[j]-pop[i])*(np.random.rand(dim)-0.5)*c
            new_pop[i] = np.clip(cand, bounds[:,0], bounds[:,1])
        new_fit = np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in new_pop])
        better = new_fit < fit
        pop[better] = new_pop[better]; fit[better] = new_fit[better]
        if fit.min() < best_f: best_f=float(fit.min()); best_p=pop[np.argmin(fit)].copy()
        print(f"  iter {it+1:02d}/{iters} best={best_f:.4f}")
    return best_p, best_f

def add_result_row(rows, tail_rows, H, split, model, y_true, y_pred, extra=None):
    s = summarize_regression(y_true, y_pred)
    row = {"H":H,"split":split,"model":model,
           "MAE":s["MAE"],"RMSE":s["RMSE"],"R2":s["R2"],"n_test":len(y_true)}
    if extra: row.update(extra)
    rows.append(row)
    for p,thr,mae,rmse,n_tail in compute_tail_metrics(y_true, y_pred):
        tr = {"H":H,"split":split,"model":model,"percentile":p,
              "threshold":float(thr),"tail_MAE":float(mae) if pd.notna(mae) else np.nan,
              "tail_RMSE":float(rmse) if pd.notna(rmse) else np.nan,"n_tail":int(n_tail)}
        if extra: tr.update(extra)
        tail_rows.append(tr)

# ============================================================

# ============================================================
# CELL 5 — LOAD NPZ, DENORMALISE, BUILD GRAPH
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)

# ── Station metadata ─────────────────────────────────────────
meta = pd.read_csv(os.path.join(CITY_PATH, "station_meta.csv"))
print("station_meta columns:", meta.columns.tolist())
for old, new in [("latitude","lat"),("longitude","lon")]:
    if old in meta.columns: meta = meta.rename(columns={old:new})
meta["station_id"] = meta["station_id"].astype(str)
station_ids = meta["station_id"].tolist()
N = len(station_ids)
print(f"Delhi: {N} stations")

# ── Norm stats ───────────────────────────────────────────────
with open(os.path.join(CITY_PATH, "norm_stats.json")) as f:
    norm_stats = json.load(f)

def get_norm_params(ns, feature, feat_idx, N):
    if feature in ns and "mean" in ns[feature]:
        mu  = np.array(ns[feature]["mean"], dtype=np.float32)
        std = np.array(ns[feature]["std"],  dtype=np.float32)
        if mu.ndim == 0:
            mu = np.full(N, float(mu), dtype=np.float32)
            std = np.full(N, float(std), dtype=np.float32)
    elif "mean" in ns:
        ma = np.array(ns["mean"], dtype=np.float32).ravel()
        sa = np.array(ns["std"],  dtype=np.float32).ravel()
        if len(ma) == len(ASTRA_FEATURES):
            mu = np.full(N, ma[feat_idx], dtype=np.float32)
            std = np.full(N, sa[feat_idx], dtype=np.float32)
        elif len(ma) == len(ASTRA_FEATURES) * N:
            mu  = ma.reshape(len(ASTRA_FEATURES), N)[feat_idx]
            std = sa.reshape(len(ASTRA_FEATURES), N)[feat_idx]
        else:
            mu = np.zeros(N, dtype=np.float32)
            std = np.ones(N, dtype=np.float32)
    else:
        mu = np.zeros(N, dtype=np.float32)
        std = np.ones(N, dtype=np.float32)
    return mu, np.where(std < 1e-6, 1.0, std)

def denorm_X(X_norm, feat_indices, ns, N):
    X_out = X_norm.copy()
    for li, gi in enumerate(feat_indices):
        mu, std = get_norm_params(ns, ASTRA_FEATURES[gi], gi, N)
        X_out[:,:,:,li] = X_norm[:,:,:,li] * std[None,None,:] + mu[None,None,:]
    return X_out

def denorm_Y(Y_norm, poll_idx, h_idx, ns, N):
    mu, std = get_norm_params(ns, ASTRA_FEATURES[poll_idx], poll_idx, N)
    return Y_norm[:, h_idx, :, poll_idx] * std[None,:] + mu[None,:]

# ── Load NPZ ─────────────────────────────────────────────────
def load_npz(split):
    d = np.load(os.path.join(CITY_PATH, f"windows_{split}.npz"))
    print(f"  {split}: keys={list(d.keys())}")
    return d

print("Loading NPZ files...")
train_npz = load_npz("train")
test_npz  = load_npz("test")

def get_arr(npz, candidates):
    for k in candidates:
        if k in npz: return npz[k]
    raise KeyError(f"None of {candidates} found. Keys: {list(npz.keys())}")

X_tr_full = get_arr(train_npz, ["X","x","features","inputs"])
Y_tr_full = get_arr(train_npz, ["Y","y","targets","labels"])
X_te_full = get_arr(test_npz,  ["X","x","features","inputs"])
Y_te_full = get_arr(test_npz,  ["Y","y","targets","labels"])

print(f"X_train {X_tr_full.shape}  Y_train {Y_tr_full.shape}")
print(f"X_test  {X_te_full.shape}  Y_test  {Y_te_full.shape}")

S_tr, L_full, N_chk, F_full = X_tr_full.shape
assert N_chk == N, f"Station mismatch: NPZ={N_chk}, meta={N}"

# ── Extract 24-step window + 6 features, denormalise ─────────
X_tr_sub = X_tr_full[:, -L:, :, :][:, :, :, FEAT_IDX].astype(np.float32)
X_te_sub = X_te_full[:, -L:, :, :][:, :, :, FEAT_IDX].astype(np.float32)
X_tr_dn  = denorm_X(X_tr_sub, FEAT_IDX, norm_stats, N)
X_te_dn  = denorm_X(X_te_sub, FEAT_IDX, norm_stats, N)
print(f"Denormalised X_train: {X_tr_dn.shape}  "
      f"pm25 range [{X_tr_dn[:,:,:,0].min():.1f}, {X_tr_dn[:,:,:,0].max():.1f}]")

fill_values = build_fill_values_from_train_timeline(
    X_tr_dn.reshape(-1, N, len(FEATURES)))

u_idx = FEATURES.index("u10")
v_idx = FEATURES.index("v10")

# ── Per-horizon data dict ─────────────────────────────────────
print("Building per-horizon data dict...")
all_data = {}
for H in H_LIST:
    h_idx = H_TO_IDX[H]
    Y_tr = denorm_Y(Y_tr_full, PM25_POLL_IDX, h_idx, norm_stats, N)
    Y_te = denorm_Y(Y_te_full, PM25_POLL_IDX, h_idx, norm_stats, N)
    M_tr = ((~np.isnan(Y_tr)) & (Y_tr >= 0)).astype(np.float32)
    M_te = ((~np.isnan(Y_te)) & (Y_te >= 0)).astype(np.float32)
    X_tr_imp = impute_windows(X_tr_dn, fill_values)
    X_te_imp = impute_windows(X_te_dn, fill_values)
    all_data[H] = {
        "X_train_imp": X_tr_imp, "X_test_imp": X_te_imp,
        "Y_train": Y_tr,  "Y_test": Y_te,
        "M_train": M_tr,  "M_test": M_te,
        "target_times_train": np.arange(len(Y_tr)),
        "target_times_test":  np.arange(len(Y_te)),
    }
    p99 = np.nanpercentile(Y_tr[M_tr.astype(bool)], 99) if M_tr.sum() > 0 else float("nan")
    print(f"  H={H:2d}h  train={len(Y_tr):6d}  test={len(Y_te):5d}  "
          f"valid_train={M_tr.sum():.0f}  pm25_99pct={p99:.1f}")

# ── Build KNN graph from station coords ──────────────────────
c = meta[["lat","lon"]].to_numpy(dtype=float)
D = np.zeros((N, N))
for i in range(N):
    D[i,:] = haversine_km_vec(c[i,0], c[i,1], c[:,0], c[:,1])
sigma = np.median(D[D > 0])
edges_list = []
for i in range(N):
    for j in np.argsort(D[i])[1:min(GRAPH_K+1, N)]:
        edges_list.append((i, j,
            float(np.exp(-(D[i,j]**2)/(2*sigma**2))),
            float(D[i,j])))
edges_df = pd.DataFrame(edges_list, columns=["src","dst","w_dist","dist_km"])

A_s = np.zeros((N,N), dtype=np.float32)
for _, r in edges_df.iterrows():
    A_s[int(r.src), int(r.dst)] = r.w_dist
rs = A_s.sum(axis=1, keepdims=True)
A_static = np.divide(A_s, rs, out=np.zeros_like(A_s), where=rs > 0)

src_arr = edges_df["src"].to_numpy(dtype=int)
dst_arr = edges_df["dst"].to_numpy(dtype=int)
w_dist  = edges_df["w_dist"].to_numpy(dtype=np.float32)
edge_bearing = bearing_radians(c[src_arr,0], c[src_arr,1],
                                c[dst_arr,0], c[dst_arr,1])
graph_ctx = {"src":src_arr, "dst":dst_arr, "w_dist":w_dist,
             "edge_bearing":edge_bearing,
             "u_idx":u_idx, "v_idx":v_idx, "N":N}
print(f"Graph: {N} nodes, {len(edges_df)} directed edges (k={GRAPH_K})")

# ============================================================
# CELL 6 — TRAINING LOOP
# ============================================================
bounds = np.array([
    [1.0,8.0],[0.01,0.20],[-5.0,-2.0],[3.0,8.0],[0.02,0.15],
    [0.60,1.00],[0.60,1.00],[0.10,5.0],[200.0,600.0],
    [1.5,4.0],[3.0,8.0],[6.0,20.0]
])

results_rows, tail_rows, tau_rows = [], [], []
all_models = {}

for H in H_LIST:
    import time as _time
    _h_start = _time.time()
    print(f"\n{'='*50}\nH={H}h  [DELHI PM2.5]  [{_time.strftime('%H:%M:%S')}]\n{'='*50}")
    pack = all_data[H]
    Xtr = pack["X_train_imp"]; Ytr = pack["Y_train"]; Mtr = pack["M_train"]
    Xte = pack["X_test_imp"];  Yte = pack["Y_test"];  Mte = pack["M_test"]

    tr, va = train_val_split_time_order(
        Xtr, Ytr, Mtr, pack["target_times_train"], frac=VALID_FRAC)

    btd, _ = choose_best_tau_dynamic(tr, va, graph_ctx, TAU_LIST)
    bts, _ = choose_best_tau_static(tr, va, A_static, TAU_LIST)
    all_models[H] = {"tau_dyn": btd, "tau_static": bts}
    print(f"tau_dyn={btd}  tau_static={bts}")

    # No-graph baselines
    for name, model in [
        ("Ridge", Pipeline([("sc",StandardScaler()),("r",Ridge(alpha=10.0))])),
        ("Huber", Pipeline([("sc",StandardScaler()),
                            ("h",HuberRegressor(epsilon=1.35,max_iter=1000))])),
        ("RF",    RandomForestRegressor(n_estimators=100, max_depth=8,
                                         n_jobs=-1, random_state=RANDOM_SEED)),
        ("XGB",   XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.07,
                                subsample=0.9, colsample_bytree=0.8,
                                tree_method="hist", n_jobs=-1,
                                random_state=RANDOM_SEED)),
    ]:
        res = fit_flat_regressor(model, Xtr,Ytr,Mtr, Xte,Yte,Mte)
        add_result_row(results_rows, tail_rows, H, "test", name,
                       res["y_true"], res["y_pred"])
        print(f"{name} done")

    # Static-graph baselines
    for name, model in [
        ("SG-Ridge", Pipeline([("sc",StandardScaler()),("r",Ridge(alpha=10.0))])),
        ("SG-Huber", Pipeline([("sc",StandardScaler()),
                                ("h",HuberRegressor(epsilon=1.35,max_iter=1000))])),
    ]:
        res = fit_static_graph_regressor(
            model, Xtr,Ytr,Mtr, Xte,Yte,Mte, A_static, bts)
        add_result_row(results_rows, tail_rows, H, "test", name,
                       res["y_true"], res["y_pred"], {"tau": bts})
        print(f"{name} done")

    # Dynamic graph
    res = fit_huber_graph(Xtr,Ytr,Mtr, Xte,Yte,Mte, graph_ctx, tau=btd)
    add_result_row(results_rows, tail_rows, H, "test", "Huber-Graph",
                   res["y_true"], res["y_pred"], {"tau": btd})
    print("Huber-Graph done")

    res = fit_huber_hybrid(Xtr,Ytr,Mtr, Xte,Yte,Mte, graph_ctx,
                           tau=btd, num_boost_round=400, seed=RANDOM_SEED)
    add_result_row(results_rows, tail_rows, H, "test", "Hybrid",
                   res["y_true"], res["y_pred_final"], {"tau": btd})
    print("Hybrid done")

    if RUN_TUNED:
        best_p, best_obj = jso_optimize(
            tr, va, graph_ctx, btd, bounds, JSO_POP, JSO_ITERS, RANDOM_SEED)
        al,ep,lha,md,eta,sub,col,lam,nr,wm,wd_w,wt = best_p
        ha = 10**float(lha)

        res = fit_huber_graph(Xtr,Ytr,Mtr, Xte,Yte,Mte, graph_ctx,
                              tau=btd, alpha=float(al), eps=float(ep),
                              huber_alpha=ha)
        add_result_row(results_rows, tail_rows, H, "test", "Tuned-Huber-Graph",
                       res["y_true"], res["y_pred"],
                       {"tau":btd, "val_obj":float(best_obj)})

        res = fit_huber_hybrid(Xtr,Ytr,Mtr, Xte,Yte,Mte, graph_ctx,
                               tau=btd, alpha=float(al), eps=float(ep),
                               huber_alpha=ha,
                               xgb_params={
                                   "objective":"reg:pseudohubererror",
                                   "max_depth":int(round(md)),
                                   "eta":float(eta),
                                   "subsample":float(sub),
                                   "colsample_bytree":float(col),
                                   "lambda":float(lam),
                                   "tree_method":"hist",
                                   "seed":RANDOM_SEED,
                                   "verbosity":0},
                               num_boost_round=int(round(nr)),
                               seed=RANDOM_SEED,
                               w_mid=float(wm), w_danger=float(wd_w),
                               w_tail=float(wt))
        add_result_row(results_rows, tail_rows, H, "test", "Tuned-Hybrid",
                       res["y_true"], res["y_pred_final"],
                       {"tau":btd, "val_obj":float(best_obj)})
        print("Tuned models done")

    print(f"H={H}h total: {(_time.time()-_h_start)/60:.1f} min")

    # Checkpoint after each horizon
    pd.DataFrame(results_rows).sort_values(["H","MAE"]).reset_index(drop=True).to_csv(
        f"{OUT_DIR}/ckpt_results_H{H}.csv", index=False)
    pd.DataFrame(tail_rows).to_csv(f"{OUT_DIR}/ckpt_tail_H{H}.csv", index=False)
    print(f"  ✓ Checkpoint saved for H={H}h")

# ============================================================
# CELL 7 — EXCEEDANCE HEAD
# ============================================================
exceed_rows = []
for H in H_LIST:
    pack = all_data[H]
    tau  = all_models[H]["tau_dyn"]
    Ztr = make_graph_features_dynamic(pack["X_train_imp"], graph_ctx, tau=tau)
    Zte = make_graph_features_dynamic(pack["X_test_imp"],  graph_ctx, tau=tau)
    Xc_tr = Ztr.reshape(-1, Ztr.shape[-1])
    Xc_te = Zte.reshape(-1, Zte.shape[-1])
    ytr = pack["Y_train"].reshape(-1)
    yte = pack["Y_test"].reshape(-1)
    mtr = pack["M_train"].reshape(-1) > 0.5
    mte = pack["M_test"].reshape(-1)  > 0.5
    thr = np.percentile(ytr[mtr], 95)
    clf = Pipeline([("sc", StandardScaler()),
                    ("logit", LogisticRegression(
                        max_iter=1000, class_weight="balanced"))])
    clf.fit(Xc_tr[mtr], (ytr[mtr] >= thr).astype(int))
    prob    = clf.predict_proba(Xc_te[mte])[:,1]
    yte_cls = (yte[mte] >= thr).astype(int)
    pr,rc,f1,_ = precision_recall_fscore_support(
        yte_cls, (prob >= 0.5).astype(int), average="binary", zero_division=0)
    ap = average_precision_score(yte_cls, prob)
    exceed_rows.append({
        "H":H, "threshold_95pct":float(thr),
        "precision":float(pr), "recall":float(rc),
        "f1":float(f1), "pr_auc":float(ap),
        "n_pos":int(yte_cls.sum()), "n_total":int(len(yte_cls))
    })

# ============================================================
# CELL 8 — SAVE AND DISPLAY
# ============================================================
results_df = pd.DataFrame(results_rows).sort_values(["H","MAE"]).reset_index(drop=True)
tail_df    = pd.DataFrame(tail_rows).sort_values(["H","model","percentile"]).reset_index(drop=True)
exceed_df  = pd.DataFrame(exceed_rows).sort_values("H").reset_index(drop=True)

results_df.to_csv(f"{OUT_DIR}/results_main.csv", index=False)
tail_df.to_csv(   f"{OUT_DIR}/results_tail.csv", index=False)
exceed_df.to_csv( f"{OUT_DIR}/exceedance.csv",   index=False)
joblib.dump(all_models, f"{OUT_DIR}/models.pkl")

print("\n=== DELHI PM2.5 RESULTS ===")
display(results_df)
print("\n=== EXCEEDANCE HEAD (Delhi winter episodes) ===")
display(exceed_df)


station_meta columns: ['station_id', 'station_name', 'lat', 'lon', 'node_idx']
Delhi: 30 stations
Loading NPZ files...
  train: keys=['X', 'Y', 'season']
  test: keys=['X', 'Y', 'season']
X_train (18340, 72, 30, 17)  Y_train (18340, 5, 30, 4)
X_test  (3922, 72, 30, 17)  Y_test  (3922, 5, 30, 4)
Denormalised X_train: (18340, 24, 30, 6)  pm25 range [0.1, 1000.0]
Building per-horizon data dict...
  H= 1h  train= 18340  test= 3922  valid_train=550200  pm25_99pct=464.0
  H= 3h  train= 18340  test= 3922  valid_train=550200  pm25_99pct=464.0
  H= 6h  train= 18340  test= 3922  valid_train=550200  pm25_99pct=464.0
  H=12h  train= 18340  test= 3922  valid_train=550200  pm25_99pct=464.0
  H=24h  train= 18340  test= 3922  valid_train=550200  pm25_99pct=464.0
Graph: 30 nodes, 120 directed edges (k=4)

H=1h  [DELHI PM2.5]  [18:03:54]
tau_dyn=2  tau_static=2
Ridge done
Huber done
RF done
XGB done
SG-Ridge done
SG-Huber done
Huber-Graph done
Hybrid done
  initial best=19.2278
  iter 01/8 best=19.1326


,H,split,model,MAE,RMSE,R2,n_test,tau,val_obj
0,1,test,Hybrid,14.908779,27.662103,0.944906,117660,2.0,NaN
1,1,test,Tuned-Hybrid,14.935237,27.604517,0.945135,117660,2.0,19.082205
2,1,test,XGB,14.991775,28.539987,0.941354,117660,NaN,NaN
3,1,test,Huber,15.040428,28.082522,0.943219,117660,NaN,NaN
4,1,test,Ridge,15.115127,27.748884,0.944560,117660,NaN,NaN
5,1,test,RF,15.270443,28.326591,0.942227,117660,NaN,NaN
6,1,test,SG-Huber,16.227504,29.841037,0.935885,117660,2.0,NaN
7,1,test,Tuned-Huber-Graph,16.229525,29.842170,0.935880,117660,2.0,19.082205
8,1,test,Huber-Graph,16.232442,29.846694,0.935860,117660,2.0,NaN
9,1,test,SG-Ridge,16.362005,29.642347,0.936736,117660,2.0,NaN



=== EXCEEDANCE HEAD (Delhi winter episodes) ===


,H,threshold_95pct,precision,recall,f1,pr_auc,n_pos,n_total
0,1,306.666656,0.560543,0.964461,0.709010,0.896509,9595,117660
1,3,306.666656,0.362231,0.902867,0.517029,0.675205,9626,117660
2,6,306.666656,0.288404,0.884607,0.434990,0.482985,9680,117660
3,12,306.666656,0.231192,0.844570,0.363013,0.309069,9715,117660
4,24,306.500000,0.271525,0.896728,0.416835,0.498225,9780,117660
